In [ ]:
!git clone https://github.com/nnathuu/diabetes-readmission-prediction.git
%cd diabetes-readmission-prediction
!ls data

Cloning into 'diabetes-readmission-prediction'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 94 (delta 33), reused 67 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 13.10 MiB | 20.18 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/diabetes-readmission-prediction
test_clean.csv	train_clean.csv  train_patient_groups.csv


In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("data/train_clean.csv")
test_df = pd.read_csv("data/test_clean.csv")

patient_groups = pd.read_csv(
    "data/train_patient_groups.csv"
)["patient_nbr"]

assert len(train_df) == len(patient_groups)
assert patient_groups.notna().all()
assert train_df.columns.equals(test_df.columns)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Patient groups:", patient_groups.shape)

train_df.head()

Train shape: (79541, 40)
Test shape : (19802, 40)
Patient groups: (79541,)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,tolazamide,insulin,glyburide-metformin,glipizide-metformin,change,diabetesMed,target,diag_1_group,diag_2_group,diag_3_group
0,Caucasian,Female,[0-10),Unknown,Unknown,1,1,Unknown,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,0,Diabetes,Missing,Missing
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,...,No,Up,No,No,Ch,Yes,0,Endocrine_other,Diabetes,Endocrine_other
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,...,No,No,No,No,No,Yes,0,Pregnancy_Childbirth,Diabetes,Supplementary_V
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,...,No,Up,No,No,Ch,Yes,0,Infectious,Diabetes,Circulatory
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,...,No,Steady,No,No,Ch,Yes,0,Neoplasms,Neoplasms,Diabetes


#Advanced feature


In [ ]:
from src.features import add_advanced_features

train_fe = add_advanced_features(train_df)
test_fe = add_advanced_features(test_df)

print("Train shape before feature engineering:", train_df.shape)
print("Train shape after feature engineering :", train_fe.shape)

print("\nTest shape before feature engineering:", test_df.shape)
print("Test shape after feature engineering :", test_fe.shape)

Train shape before feature engineering: (79541, 40)
Train shape after feature engineering : (79541, 47)

Test shape before feature engineering: (19802, 40)
Test shape after feature engineering : (19802, 47)


## Baseline Models

Before training the baseline models, the target variable is separated from
the predictors. Numerical features are standardized, while categorical
features are transformed using One-Hot Encoding.

The preprocessing pipeline is fitted only on the training data to prevent
data leakage.

In [ ]:
# Separate features and target
X_train = train_fe.drop(columns=["target"])
y_train = train_fe["target"]

X_test = test_fe.drop(columns=["target"])
y_test = test_fe["target"]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (79541, 46)
X_test : (19802, 46)
y_train: (79541,)
y_test : (19802,)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = patient_groups.to_numpy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

fit_idx, val_idx = next(
    gss.split(X_train, y_train, groups=groups)
)

X_fit = X_train.iloc[fit_idx]
X_val = X_train.iloc[val_idx]

y_fit = y_train.iloc[fit_idx]
y_val = y_train.iloc[val_idx]

groups_fit = groups[fit_idx]
groups_val = groups[val_idx]

shared_patients = len(set(groups_fit) & set(groups_val))
assert shared_patients == 0

print("Fit:", X_fit.shape)
print("Validation:", X_val.shape)
print("Shared patients:", shared_patients)
print("Fit positive rate:", round(y_fit.mean(), 4))
print("Validation positive rate:", round(y_val.mean(), 4))

Fit: (63807, 46)
Validation: (15734, 46)
Shared patients: 0
Fit positive rate: 0.1143
Validation positive rate: 0.1145


In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_ids = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in categorical_ids:
    if col in X_train.columns and col not in categorical_features:
        categorical_features.append(col)

numerical_features = [
    col for col in X_train.select_dtypes(include="number").columns
    if col not in categorical_features
]

assert set(numerical_features + categorical_features) == set(X_train.columns)

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_previous_visits', 'has_prior_inpatient', 'has_prior_emergency', 'medications_per_day', 'labs_per_day', 'procedures_per_day', 'treatment_intensity']
Categorical features: ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'change', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("scaler", RobustScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


### Baseline Model Training

Three baseline classification models are trained:

- Logistic Regression
- Decision Tree
- Random Forest

No class balancing or hyperparameter tuning is applied at this stage.
The baseline results will be used as a reference for evaluating more advanced approaches.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

import pandas as pd

**Logistic Regression**

In [ ]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

**Decision Tree**

In [ ]:
decision_tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

**Random Forest**

In [ ]:
random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

## Advanced Models

Advanced gradient boosting models are evaluated to capture more complex
non-linear relationships in the data.


In [ ]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.1 MB/s eta 0:00:00


**CatBoost**

In [ ]:
from catboost import CatBoostClassifier

catboost_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function="Logloss",
        random_seed=42,
        verbose=0
    ))
])

## Imbalance Handling

The target variable is highly imbalanced, with the positive class representing
only about 11% of the training data.

Two imbalance handling strategies are evaluated:

- Class weighting
- SMOTE (Synthetic Minority Over-sampling Technique)

Their performance is compared with the original models using Recall, F1-score,
ROC-AUC, and PR-AUC.

**Class Weight**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logistic_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
from catboost import CatBoostClassifier

catboost_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function="Logloss",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=0
    ))
])

**SMOTE**

In [ ]:
!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
logistic_smote_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(
        random_state=42
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Model Selection with Group-Aware Cross-Validation



In [ ]:
from sklearn.base import clone

candidates = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline,
    "CatBoost": catboost_pipeline,

    "Logistic Regression + Class Weight": logistic_balanced_pipeline,
    "Decision Tree + Class Weight": decision_tree_balanced_pipeline,
    "Random Forest + Class Weight": random_forest_balanced_pipeline,
    "CatBoost + Class Weight": catboost_balanced_pipeline,
    "Logistic Regression + SMOTE": logistic_smote_pipeline,
}

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)
cv_rows = []
for name, pipeline in candidates.items():
    scores = cross_val_score(
        clone(pipeline),
        X_fit,
        y_fit,
        groups=groups_fit,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "Mean CV AP": scores.mean(),
        "Std CV AP": scores.std()
    })
cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values("Mean CV AP", ascending=False)
    .reset_index(drop=True)
)

display(cv_results.round(4))

,Model,Mean CV AP,Std CV AP
0,CatBoost,0.2267,0.0033
1,CatBoost + Class Weight,0.2189,0.0029
2,Logistic Regression,0.2108,0.0049
3,Logistic Regression + Class Weight,0.2094,0.0049
4,Random Forest,0.2066,0.0045
5,Random Forest + Class Weight,0.2065,0.0037
6,Logistic Regression + SMOTE,0.2006,0.0031
7,Decision Tree,0.1235,0.0029
8,Decision Tree + Class Weight,0.1200,0.0023


In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

validation_rows = []

for name, pipeline in candidates.items():
    print("Đang chạy:", name, flush=True)

    model = clone(pipeline)
    model.fit(X_fit, y_fit)

    positive_col = list(model.classes_).index(1)
    val_proba = model.predict_proba(X_val)[:, positive_col]
    val_pred = (val_proba >= 0.5).astype(int)

    validation_rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_val, val_proba),
        "AP": average_precision_score(y_val, val_proba),
        "Precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, val_pred, zero_division=0
        )
    })

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values("AP", ascending=False)
    .reset_index(drop=True)
)

display(validation_results.round(4))

Đang chạy: Logistic Regression
Đang chạy: Decision Tree


On the validation set, CatBoost achieved the highest ranking performance (ROC-AUC 0.678, AP 0.239), although the differences among the top boosting models and Logistic Regression are small (AP between 0.221 and 0.239). Class weighting and SMOTE do not improve the ranking metrics; they mainly shift the operating point. At the default threshold of 0.5, models without balancing detect almost no positive cases (recall 0.1%–2.3%), so their high precision is based on very few predictions, whereas class-weighted models reach recall of about 0.50–0.56 at precision of about 0.18–0.19. SMOTE reduced both ROC-AUC and AP of Logistic Regression, and the unconstrained Decision Tree performed close to the random baseline (AP 0.123 vs prevalence 0.115).

In [23]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

best_pipeline = candidates["CatBoost"]

param_dist = {
    "classifier__depth": randint(4, 10),
    "classifier__learning_rate": uniform(0.01, 0.2),
    "classifier__l2_leaf_reg": uniform(1, 10),
    "classifier__iterations": randint(200, 800),
}

cv_search = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=clone(best_pipeline),
    param_distributions=param_dist,
    n_iter=8,
    scoring="average_precision",
    cv=cv_search,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(
    X_fit,
    y_fit,
    groups=groups_fit
)

print("Best params:", search.best_params_)
print("Best CV AP:", search.best_score_)

tuned_model = search.best_estimator_

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params: {'classifier__depth': 8, 'classifier__iterations': 543, 'classifier__l2_leaf_reg': np.float64(9.324426408004218), 'classifier__learning_rate': np.float64(0.052467822135655234)}
Best CV AP: 0.22834274766855256


In [24]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
best_params = {
    key.removeprefix("classifier__"): (
        value.item() if isinstance(value, np.generic) else value
    )
    for key, value in search.best_params_.items()
}

def make_final_pipeline():
    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", CatBoostClassifier(
            **best_params,
            loss_function="Logloss",
            random_seed=42,
            verbose=0
        ))
    ])

val_fit = make_final_pipeline()
val_fit.fit(X_fit, y_fit)

positive_col = list(val_fit.classes_).index(1)
val_proba = val_fit.predict_proba(X_val)[:, positive_col]

precision, recall, thresholds = precision_recall_curve(
    y_val, val_proba, pos_label=1
)

denom = precision[:-1] + recall[:-1]
f1_scores = np.divide(
    2 * precision[:-1] * recall[:-1],
    denom,
    out=np.zeros_like(denom),
    where=denom != 0
)

best_threshold = float(thresholds[np.argmax(f1_scores)])
print(f"Selected threshold: {best_threshold:.4f}")
final_model = make_final_pipeline()
final_model.fit(X_train, y_train)

positive_col = list(final_model.classes_).index(1)
test_proba = final_model.predict_proba(X_test)[:, positive_col]
y_pred_final = (test_proba >= best_threshold).astype(int)

print("\n=== FINAL TEST RESULTS ===")
print("Model: Tuned CatBoost")
print(f"Threshold: {best_threshold:.4f}")
print(f"Test AP: {average_precision_score(y_test, test_proba):.4f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}")
print(classification_report(
    y_test,
    y_pred_final,
    digits=4,
    zero_division=0
))

print("Confusion matrix:\n", confusion_matrix(
    y_test, y_pred_final, labels=[0, 1]
))

Selected threshold: 0.1366

=== FINAL TEST RESULTS ===
Model: Tuned CatBoost
Threshold: 0.1366
Test AP: 0.2444
Test ROC-AUC: 0.6787
              precision    recall  f1-score   support

           0     0.9201    0.7728    0.8400     17586
           1     0.2059    0.4675    0.2859      2216

    accuracy                         0.7386     19802
   macro avg     0.5630    0.6201    0.5630     19802
weighted avg     0.8402    0.7386    0.7780     19802

Confusion matrix:
 [[13590  3996]
 [ 1180  1036]]


### Final Results and Interpretation

The tuned CatBoost model was refitted on the full training set
and evaluated on the held-out test set. A threshold of 0.1360
was selected by maximizing F1 on the patient-disjoint validation
set.

- **Average Precision (AP): 0.2453**, above the test positive
  prevalence of approximately 0.112. This indicates useful
  predictive signal, although performance remains limited.
- **ROC-AUC: 0.6788**, indicating moderate discrimination between
  encounters with and without readmission within 30 days.
- **Positive-class precision: 0.2048**. Approximately one in five
  encounters flagged by the model resulted in readmission
  within 30 days.
- **Positive-class recall: 0.4702**. The model identified
  approximately 47% of actual positive encounters.
- **Positive-class F1: 0.2854**, reflecting the trade-off between
  precision and recall at the selected threshold.

The confusion matrix shows 1,042 true positives, 1,174 false
negatives, 4,045 false positives, and 13,541 true negatives.
The model therefore misses more than half of positive encounters
and produces many false alerts.

Accuracy is not the primary evaluation metric because the
dataset is imbalanced. Predicting every encounter as negative
would achieve approximately 88.8% accuracy but zero positive-class
recall.

Patient-level splitting reduces leakage from repeated encounters
of the same patient. However, some preprocessing steps were
learned from the full training set before internal validation
and cross-validation. In addition, the validation-selected
threshold may behave differently after refitting on all training
data.

Overall, the model performs better than the AP reference level
but has substantial limitations. These results support an
academic comparison of methods rather than clinical deployment.

In [25]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(output_dir / "cv_results.csv", index=False)

pd.DataFrame({
    "y_true": np.asarray(y_test),
    "probability": test_proba,
    "prediction": y_pred_final
}).to_csv(output_dir / "test_predictions.csv", index=False)